# 03 — Data Cleaning & Preparation

## 1. Introduction and objectives

Phase 1A (`01_data_exploration.ipynb`) described what the Olist tables contain, and Phase 1B (`02_data_quality_validation.ipynb`) tested whether their values and relationships are consistent. Phase 1C turns those findings into **prepared copies** of the source tables that later phases can rely on: feature engineering, synthetic contact generation, contact-volume forecasting, and the digital-twin simulation.

This notebook:

1. Loads the seven Olist source tables without modifying `data/raw/`.
2. Investigates the issues reported in Phase 1B (repeated identifiers, timestamp order, missing values, orders without related records).
3. Creates prepared copies with clear data types and documented data-quality flags.
4. Re-validates the prepared copies and exports them to `data/processed/`.
5. Records every decision and every unresolved issue.

This notebook does **not** impute values, delete records, correct timestamps, join everything into one table, or generate contacts.

> **Olist is e-commerce data, not contact-center data.** It contains no customer contact logs, contact reasons, call durations, queue waiting times, or agent assignments. Those variables will be **simulated in a later phase** using explicit, documented assumptions. Nothing in this notebook creates or implies them.

## 2. Imports and project paths

Same path convention as the earlier notebooks: the notebook works whether it is started from the project root or from the `notebooks/` folder.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_DIR, "exists:", DATA_DIR.exists())
print("Processed data directory:", PROCESSED_DIR, "exists:", PROCESSED_DIR.exists())
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory missing: {DATA_DIR}. Put Olist CSVs in data/raw/olist/.")

Project root: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Raw data directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\raw\olist exists: True
Processed data directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\processed exists: True


## 3. Load the source data

We load the seven tables used in Phase 1B. If any file is missing, the notebook stops with a clear message instead of continuing with incomplete or invented data.

**One loading choice:** the ZIP-code prefix columns are stored in the CSV files as quoted 5-digit text with leading zeros (for example `"09790"`). By default pandas would read them as numbers and drop the leading zero. We tell pandas to read these two columns as text so the values stay exactly as written in the file. All other columns use pandas' default types.

We also compute a **fingerprint** of each loaded table: one number calculated from every value. At the end of the notebook we recompute it to prove that the original DataFrames were never changed.

In [2]:
SOURCE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
}

# Columns that must be read as text to keep leading zeros
TEXT_COLUMNS = {
    "customers": {"customer_zip_code_prefix": str},
    "sellers": {"seller_zip_code_prefix": str},
}

missing_files = []
for dataset_name, file_name in SOURCE_FILES.items():
    if not (DATA_DIR / file_name).exists():
        missing_files.append(file_name)
if missing_files:
    raise FileNotFoundError(
        f"Missing Olist files in {DATA_DIR}: {missing_files}. "
        "Download the Olist dataset into data/raw/olist/ before running this notebook."
    )

raw_datasets = {}
for dataset_name, file_name in SOURCE_FILES.items():
    column_types = TEXT_COLUMNS.get(dataset_name)  # None means pandas default types
    raw_datasets[dataset_name] = pd.read_csv(DATA_DIR / file_name, dtype=column_types)

load_summary = []
for dataset_name, dataset in raw_datasets.items():
    load_summary.append({"dataset": dataset_name, "file": SOURCE_FILES[dataset_name],
                         "rows": len(dataset), "columns": len(dataset.columns)})
display(pd.DataFrame(load_summary))

,dataset,file,rows,columns
0,orders,olist_orders_dataset.csv,99441,8
1,order_items,olist_order_items_dataset.csv,112650,7
2,customers,olist_customers_dataset.csv,99441,5
3,payments,olist_order_payments_dataset.csv,103886,5
4,reviews,olist_order_reviews_dataset.csv,99224,7
5,products,olist_products_dataset.csv,32951,9
6,sellers,olist_sellers_dataset.csv,3095,4


In [3]:
# Fingerprint of each raw table, used at the end to confirm nothing changed
raw_fingerprints = {}
for dataset_name, dataset in raw_datasets.items():
    raw_fingerprints[dataset_name] = pd.util.hash_pandas_object(dataset, index=True).sum()

# Short names for the raw tables
orders_raw = raw_datasets["orders"]
order_items_raw = raw_datasets["order_items"]
customers_raw = raw_datasets["customers"]
payments_raw = raw_datasets["payments"]
reviews_raw = raw_datasets["reviews"]
products_raw = raw_datasets["products"]
sellers_raw = raw_datasets["sellers"]

## 4. Cleaning methodology

The following principles guide every step:

- **Preserve the original source data.** Files in `data/raw/` are only read, never written.
- **Work on copies.** All transformations are applied to copies of the loaded DataFrames; the `*_raw` DataFrames stay unchanged.
- **No deletion without documented justification.** In this phase no record is removed.
- **No automatic imputation of missing timestamps.** A missing delivery date often means the event did not happen (canceled or in-progress orders).
- **Missing is not the same as invalid.** An empty review title is normal; a delivery before the purchase is suspicious. They are handled differently.
- **Investigate before correcting.** Suspicious records are counted and shown with examples first.
- **Preserve original timestamp values** when the data do not clearly support a correction. Suspicious records are *flagged*, not changed.
- **Keep source tables separate.** Joining one-to-many tables (orders → items, payments, reviews) multiplies order rows, so each table is prepared and exported on its own.
- **Record unresolved issues and assumptions** in an issue log for later phases.

**Column naming for added columns**

| Prefix | Meaning |
|---|---|
| `has_*` | `True` when a value or related record is present. Describes availability, not quality. |
| `flag_*` | `True` when the record matches a suspicious pattern worth investigating. A flag does **not** mean the record is wrong or unusable. |

Chronology flags are `True` only when *both* timestamps exist and are in an unexpected order; they are `False` when either timestamp is missing (the `has_*` columns tell those cases apart). The late-delivery column uses pandas' nullable boolean type so that orders without an actual delivery date are `<NA>` (unknown) instead of `False`.

In [4]:
# Prepared copies: every transformation below is applied to these, never to *_raw
orders_prepared = orders_raw.copy()
order_items_prepared = order_items_raw.copy()
customers_prepared = customers_raw.copy()
payments_prepared = payments_raw.copy()
reviews_prepared = reviews_raw.copy()
products_prepared = products_raw.copy()
sellers_prepared = sellers_raw.copy()

prepared_datasets = {
    "orders": orders_prepared,
    "order_items": order_items_prepared,
    "customers": customers_prepared,
    "payments": payments_prepared,
    "reviews": reviews_prepared,
    "products": products_prepared,
    "sellers": sellers_prepared,
}

# Every finding and decision is added to this list and shown in the final summary
issue_log = []

## 5. Identifiers and duplicate records

### 5.1 Exact duplicate rows

An exact duplicate is a row identical to another row in every column.

In [5]:
duplicate_rows = []
for dataset_name, dataset in raw_datasets.items():
    duplicate_count = int(dataset.duplicated().sum())
    duplicate_rows.append({"dataset": dataset_name, "rows": len(dataset), "exact_duplicate_rows": duplicate_count})
    issue_log.append({"area": "Duplicates", "dataset": dataset_name, "issue": "Exact duplicate rows",
                      "affected_records": duplicate_count, "decision": "Reported; no rows removed."})
display(pd.DataFrame(duplicate_rows))

,dataset,rows,exact_duplicate_rows
0,orders,99441,0
1,order_items,112650,0
2,customers,99441,0
3,payments,103886,0
4,reviews,99224,0
5,products,32951,0
6,sellers,3095,0


### 5.2 Uniqueness of expected identifiers

For each table we check the column (or column combination) expected to identify one record. Some identifiers are *expected* to repeat, for example `customer_unique_id`, which identifies a person who can place several orders.

In [6]:
key_checks = [
    ("orders", ["order_id"], "One row per order"),
    ("orders", ["customer_id"], "Olist issues a new customer_id for each order"),
    ("customers", ["customer_id"], "One row per customer_id"),
    ("customers", ["customer_unique_id"], "May repeat: one person, several orders"),
    ("order_items", ["order_id", "order_item_id"], "One row per item line"),
    ("payments", ["order_id", "payment_sequential"], "One row per payment step"),
    ("reviews", ["review_id"], "Expected unique; Phase 1B found repeats"),
    ("reviews", ["review_id", "order_id"], "Candidate composite key"),
    ("products", ["product_id"], "One row per product"),
    ("sellers", ["seller_id"], "One row per seller"),
]

key_results = []
for dataset_name, key_columns, expectation in key_checks:
    dataset = raw_datasets[dataset_name]
    repeated_rows = int(dataset.duplicated(subset=key_columns, keep=False).sum())
    key_results.append({
        "dataset": dataset_name,
        "key": ", ".join(key_columns),
        "expectation": expectation,
        "rows": len(dataset),
        "distinct_keys": len(dataset.drop_duplicates(subset=key_columns)),
        "rows_with_repeated_key": repeated_rows,
        "is_unique": repeated_rows == 0,
    })
key_results_df = pd.DataFrame(key_results)
display(key_results_df)

,dataset,key,expectation,rows,distinct_keys,rows_with_repeated_key,is_unique
0,orders,order_id,One row per order,99441,99441,0,True
1,orders,customer_id,Olist issues a new customer_id for each order,99441,99441,0,True
2,customers,customer_id,One row per customer_id,99441,99441,0,True
3,customers,customer_unique_id,"May repeat: one person, several orders",99441,96096,6342,False
4,order_items,"order_id, order_item_id",One row per item line,112650,112650,0,True
5,payments,"order_id, payment_sequential",One row per payment step,103886,103886,0,True
6,reviews,review_id,Expected unique; Phase 1B found repeats,99224,98410,1603,False
7,reviews,"review_id, order_id",Candidate composite key,99224,99224,0,True
8,products,product_id,One row per product,32951,32951,0,True
9,sellers,seller_id,One row per seller,3095,3095,0,True


### 5.3 `customer_id` versus `customer_unique_id`

In Olist, `customer_id` is created per order, so it is unique in both the orders and customers tables. The person behind the order is `customer_unique_id`. Any later analysis of repeat customers (for example, a customer contacting support about several orders) must use `customer_unique_id`.

In [7]:
orders_per_person = customers_raw["customer_unique_id"].value_counts()

people_summary = pd.DataFrame({
    "description": ["Distinct people (customer_unique_id)", "People with exactly 1 order",
                    "People with 2 orders", "People with 3 or more orders"],
    "count": [len(orders_per_person), int((orders_per_person == 1).sum()),
              int((orders_per_person == 2).sum()), int((orders_per_person >= 3).sum())],
})
display(people_summary)

issue_log.append({"area": "Identifiers", "dataset": "customers",
                  "issue": "customer_unique_id repeats across orders (repeat customers)",
                  "affected_records": int((orders_per_person > 1).sum()),
                  "decision": "Expected behaviour; use customer_unique_id to identify people."})

,description,count
0,Distinct people (customer_unique_id),96096
1,People with exactly 1 order,93099
2,People with 2 orders,2745
3,People with 3 or more orders,252


### 5.4 Repeated `review_id` values

Phase 1B found repeated `review_id` values. A repeat is not automatically an error, so we look at what the repeated rows have in common:

- Do they point to different orders?
- Is the review content (score, title, message, dates) the same in every copy?
- Do the orders belong to the same person?

In [8]:
repeated_review_mask = reviews_raw.duplicated(subset=["review_id"], keep=False)
repeated_reviews = reviews_raw[repeated_review_mask].copy()
print("Rows with a repeated review_id:", len(repeated_reviews))
print("Distinct repeated review_id values:", repeated_reviews["review_id"].nunique())

# Look up the person behind each order: order_id -> customer_id -> customer_unique_id
customer_by_order = orders_raw.set_index("order_id")["customer_id"]
person_by_customer = customers_raw.set_index("customer_id")["customer_unique_id"]
repeated_reviews["customer_unique_id"] = repeated_reviews["order_id"].map(customer_by_order).map(person_by_customer)

# Replace empty text with a marker only for this comparison, so empty values count as equal
comparison = repeated_reviews.fillna({"review_comment_title": "<empty>", "review_comment_message": "<empty>"})

review_groups = comparison.groupby("review_id").agg(
    rows=("order_id", "size"),
    distinct_orders=("order_id", "nunique"),
    distinct_people=("customer_unique_id", "nunique"),
    distinct_scores=("review_score", "nunique"),
    distinct_titles=("review_comment_title", "nunique"),
    distinct_messages=("review_comment_message", "nunique"),
    distinct_creation_dates=("review_creation_date", "nunique"),
    distinct_answer_times=("review_answer_timestamp", "nunique"),
)

same_content = (
    (review_groups["distinct_scores"] == 1)
    & (review_groups["distinct_titles"] == 1)
    & (review_groups["distinct_messages"] == 1)
    & (review_groups["distinct_creation_dates"] == 1)
    & (review_groups["distinct_answer_times"] == 1)
)

review_id_summary = pd.DataFrame({
    "question": [
        "Repeated review_id values",
        "...where every copy points to a different order",
        "...where all copies have identical review content",
        "...where all orders belong to the same person",
    ],
    "count": [
        len(review_groups),
        int((review_groups["distinct_orders"] == review_groups["rows"]).sum()),
        int(same_content.sum()),
        int((review_groups["distinct_people"] == 1).sum()),
    ],
})
display(review_id_summary)
print("Copies per repeated review_id:")
display(review_groups["rows"].value_counts().rename("review_ids").to_frame())

Rows with a repeated review_id: 1603
Distinct repeated review_id values: 789


,question,count
0,Repeated review_id values,789
1,...where every copy points to a different order,789
2,...where all copies have identical review content,789
3,...where all orders belong to the same person,789


Copies per repeated review_id:


,review_ids
rows,
2,764
3,25


In [9]:
# Representative examples: the first few repeated review_ids, all their rows
example_review_ids = review_groups.index[:4]
example_columns = ["review_id", "order_id", "customer_unique_id", "review_score",
                   "review_creation_date", "review_answer_timestamp", "review_comment_message"]
examples = repeated_reviews[repeated_reviews["review_id"].isin(example_review_ids)]
display(examples.sort_values(["review_id", "order_id"])[example_columns])

,review_id,order_id,customer_unique_id,review_score,review_creation_date,review_answer_timestamp,review_comment_message
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,f30856ad31d3e74253a3f4ccef670648,1,2018-03-07 00:00:00,2018-03-20 18:08:23,"O cartucho ""original HP"" 60XL não é reconhecid..."
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,f30856ad31d3e74253a3f4ccef670648,1,2018-03-07 00:00:00,2018-03-20 18:08:23,"O cartucho ""original HP"" 60XL não é reconhecid..."
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,2017-09-21 00:00:00,2017-09-26 03:27:47,NaN
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,2017-09-21 00:00:00,2017-09-26 03:27:47,NaN
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,2018-03-07 00:00:00,2018-03-08 03:00:53,Produto entregue dentro de embalagem do fornec...
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,2018-03-07 00:00:00,2018-03-08 03:00:53,Produto entregue dentro de embalagem do fornec...
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,d5ca668597a3b40c1e527db8c761f03c,5,2018-03-02 00:00:00,2018-03-05 01:43:30,NaN
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,d5ca668597a3b40c1e527db8c761f03c,5,2018-03-02 00:00:00,2018-03-05 01:43:30,NaN


**Interpretation.** Check the table above: if every repeated `review_id` has identical content and all its orders belong to the same person, the most plausible explanation is that **one review was attached to several orders of the same customer** (for example, orders placed together). This is a hypothesis supported by the pattern, not confirmed by Olist documentation.

**Decision:**

- Keep every review row. Removing copies would lose the link between the review and some of its orders.
- Do not create new IDs or change `review_id`.
- Use **`(review_id, order_id)`** as the record key of the reviews table (unique, see 5.2).
- Add `flag_repeated_review_id` so later phases can avoid counting one review several times (for example, when computing satisfaction per person).

In [10]:
issue_log.append({"area": "Identifiers", "dataset": "reviews",
                  "issue": "review_id repeated across rows (identical content, same person)",
                  "affected_records": len(repeated_reviews),
                  "decision": "Kept all rows; key is (review_id, order_id); flagged flag_repeated_review_id."})

### 5.5 Orders with more than one review

The reverse situation also exists: one order can have several review rows.

In [11]:
reviews_per_order = reviews_raw["order_id"].value_counts()
orders_with_several_reviews = reviews_per_order[reviews_per_order > 1].index
print("Orders with more than one review row:", len(orders_with_several_reviews))
print("Review rows belonging to these orders:", int(reviews_raw["order_id"].isin(orders_with_several_reviews).sum()))

example_order_ids = orders_with_several_reviews[:3]
example = reviews_raw[reviews_raw["order_id"].isin(example_order_ids)]
display(example.sort_values(["order_id", "review_creation_date"])[
    ["order_id", "review_id", "review_score", "review_creation_date", "review_answer_timestamp"]])

issue_log.append({"area": "Identifiers", "dataset": "reviews", "issue": "Orders with more than one review row",
                  "affected_records": len(orders_with_several_reviews),
                  "decision": "Kept all rows; flagged flag_order_has_multiple_reviews. Which review is final is unresolved."})

Orders with more than one review row: 547
Review rows belonging to these orders: 1098


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
44694,8e17072ec97ce29f0e1f111e598b0c85,67c2557eb0bd72e3ece1e03477c9dff5,1,2018-04-07 00:00:00,2018-04-08 22:48:27
64510,8e17072ec97ce29f0e1f111e598b0c85,2d6ac45f859465b5c185274a1c929637,1,2018-04-07 00:00:00,2018-04-07 21:13:05
92300,8e17072ec97ce29f0e1f111e598b0c85,6e4c4086d9611ae4cc0cc65a262751fe,1,2018-04-14 00:00:00,2018-04-16 11:37:31
89360,c88b1d1b157a9999ce368f218a407141,fb96ea2ef8cce1c888f4d45c8e22b793,5,2017-07-21 00:00:00,2017-07-26 13:45:15
1985,c88b1d1b157a9999ce368f218a407141,ffb8cff872a625632ac983eb1f88843c,3,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,c88b1d1b157a9999ce368f218a407141,202b5f44d09cd3cfc0d6bd12f01b044c,5,2017-07-22 00:00:00,2017-07-26 13:40:22
13982,df56136b8031ecd28e200bb18e6ddb2e,72a1098d5b410ae50fbc0509d26daeb9,5,2017-02-07 00:00:00,2017-02-10 10:46:09
2952,df56136b8031ecd28e200bb18e6ddb2e,c444278834184f72b1484dfe47de7f97,5,2017-02-08 00:00:00,2017-02-14 13:58:48
62728,df56136b8031ecd28e200bb18e6ddb2e,44f3e54834d23c5570c1d010824d4d59,5,2017-02-09 00:00:00,2017-02-09 09:07:28


**Which identifiers identify records?**

| Table | Record key | Notes |
|---|---|---|
| orders | `order_id` | Unique. |
| customers | `customer_id` | Unique, but one per order. Use `customer_unique_id` for people. |
| order_items | `(order_id, order_item_id)` | Unique; `order_id` alone repeats by design. |
| payments | `(order_id, payment_sequential)` | Unique; an order can be paid in several steps. |
| reviews | `(review_id, order_id)` | `review_id` alone repeats; see 5.4. |
| products | `product_id` | Unique. |
| sellers | `seller_id` | Unique. |

The table above summarises the checks in 5.2; check `is_unique` there if the source files change.

## 6. Timestamp inconsistencies

### 6.1 Convert timestamp columns to datetime

All Olist timestamps use the format `YYYY-MM-DD HH:MM:SS`. We parse them with that explicit format. `errors="coerce"` turns a value that cannot be parsed into `NaT` (missing) instead of stopping the notebook, so we then count non-empty values that failed. The original text stays available in the `*_raw` DataFrames.

Besides the five order milestones, we also parse the item shipping limit and the two review dates, because their intended type is clearly a date.

In [12]:
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"

ORDER_TIMESTAMP_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

timestamp_columns_to_parse = []
for column in ORDER_TIMESTAMP_COLUMNS:
    timestamp_columns_to_parse.append(("orders", column))
timestamp_columns_to_parse.append(("order_items", "shipping_limit_date"))
timestamp_columns_to_parse.append(("reviews", "review_creation_date"))
timestamp_columns_to_parse.append(("reviews", "review_answer_timestamp"))

parsing_results = []
for dataset_name, column in timestamp_columns_to_parse:
    raw_values = raw_datasets[dataset_name][column]
    parsed_values = pd.to_datetime(raw_values, format=TIMESTAMP_FORMAT, errors="coerce")
    prepared_datasets[dataset_name][column] = parsed_values

    failed = raw_values.notna() & parsed_values.isna()
    parsing_results.append({"dataset": dataset_name, "column": column,
                            "non_empty_values": int(raw_values.notna().sum()),
                            "parse_failures": int(failed.sum())})
    if failed.any():
        print(f"Values that failed to parse in {dataset_name}.{column}:")
        display(raw_datasets[dataset_name].loc[failed].head())

parsing_results_df = pd.DataFrame(parsing_results)
display(parsing_results_df)

issue_log.append({"area": "Timestamps", "dataset": "orders, order_items, reviews",
                  "issue": "Non-empty timestamps that failed to parse",
                  "affected_records": int(parsing_results_df["parse_failures"].sum()),
                  "decision": "Converted to datetime; failures (if any) shown above and kept in raw data."})

,dataset,column,non_empty_values,parse_failures
0,orders,order_purchase_timestamp,99441,0
1,orders,order_approved_at,99281,0
2,orders,order_delivered_carrier_date,97658,0
3,orders,order_delivered_customer_date,96476,0
4,orders,order_estimated_delivery_date,99441,0
5,order_items,shipping_limit_date,112650,0
6,reviews,review_creation_date,99224,0
7,reviews,review_answer_timestamp,99224,0


### 6.2 Chronology checks

The expected order of milestones is: purchase → approval → carrier handoff → customer delivery. For each pair we compare only orders where both timestamps exist. A violation is worth investigating but does not tell us *which* timestamp is wrong, so nothing is swapped or removed. Each check becomes a `flag_*` column in the prepared orders table, keyed by `order_id` so every flagged record can be traced back.

The check "delivered before purchase" is the same as a **negative purchase-to-delivery duration**.

In [13]:
# (flag name, column expected to be earlier, column expected to be later)
chronology_checks = [
    ("flag_approved_before_purchase", "order_purchase_timestamp", "order_approved_at"),
    ("flag_carrier_before_approval", "order_approved_at", "order_delivered_carrier_date"),
    ("flag_delivered_before_carrier", "order_delivered_carrier_date", "order_delivered_customer_date"),
    ("flag_delivered_before_purchase", "order_purchase_timestamp", "order_delivered_customer_date"),
]

chronology_results = []
for flag_name, earlier_column, later_column in chronology_checks:
    both_present = orders_prepared[earlier_column].notna() & orders_prepared[later_column].notna()
    reversed_order = both_present & (orders_prepared[later_column] < orders_prepared[earlier_column])
    orders_prepared[flag_name] = reversed_order

    # How far apart are the two timestamps in the reversed cases?
    gap_hours = (orders_prepared.loc[reversed_order, earlier_column]
                 - orders_prepared.loc[reversed_order, later_column]).dt.total_seconds() / 3600
    chronology_results.append({
        "flag": flag_name,
        "expected": f"{earlier_column} <= {later_column}",
        "comparable_orders": int(both_present.sum()),
        "violations": int(reversed_order.sum()),
        "percent_of_comparable": round(reversed_order.sum() / both_present.sum() * 100, 3),
        "median_gap_hours": round(gap_hours.median(), 1) if len(gap_hours) else None,
        "max_gap_hours": round(gap_hours.max(), 1) if len(gap_hours) else None,
    })
    issue_log.append({"area": "Timestamps", "dataset": "orders", "issue": f"Chronology: {flag_name}",
                      "affected_records": int(reversed_order.sum()),
                      "decision": "Flagged; original timestamps kept; not swapped or removed."})

chronology_results_df = pd.DataFrame(chronology_results)
display(chronology_results_df)

,flag,expected,comparable_orders,violations,percent_of_comparable,median_gap_hours,max_gap_hours
0,flag_approved_before_purchase,order_purchase_timestamp <= order_approved_at,99281,0,0.000,NaN,NaN
1,flag_carrier_before_approval,order_approved_at <= order_delivered_carrier_date,97644,1359,1.392,17.2,4109.3
2,flag_delivered_before_carrier,order_delivered_carrier_date <= order_delivere...,96475,23,0.024,39.9,386.3
3,flag_delivered_before_purchase,order_purchase_timestamp <= order_delivered_cu...,96476,0,0.000,NaN,NaN


In [14]:
# Representative examples for each check that has violations
for flag_name, earlier_column, later_column in chronology_checks:
    flagged_orders = orders_prepared[orders_prepared[flag_name]]
    if flagged_orders.empty:
        print(f"{flag_name}: no violations")
        continue
    print(f"{flag_name}: {len(flagged_orders)} orders, first examples")
    display(flagged_orders[["order_id", "order_status", earlier_column, later_column]].head(5))

flag_approved_before_purchase: no violations
flag_carrier_before_approval: 1359 orders, first examples


,order_id,order_status,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00


flag_delivered_before_carrier: 23 orders, first examples


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41


flag_delivered_before_purchase: no violations


In [15]:
# Negative purchase-to-delivery duration, computed explicitly
purchase_to_delivery_days = (orders_prepared["order_delivered_customer_date"]
                             - orders_prepared["order_purchase_timestamp"]).dt.total_seconds() / 86400
print("Orders with a purchase-to-delivery duration:", int(purchase_to_delivery_days.notna().sum()))
print("Negative durations:", int((purchase_to_delivery_days < 0).sum()))
display(purchase_to_delivery_days.describe().round(2).to_frame("purchase_to_delivery_days"))

Orders with a purchase-to-delivery duration: 96476
Negative durations: 0


,purchase_to_delivery_days
count,96476.00
mean,12.56
std,9.55
min,0.53
25%,6.77
50%,10.22
75%,15.72
max,209.63


**Carrier handoff before approval** is the largest group. Look at the gap distribution: if most gaps are a few hours to about a day, a plausible explanation is that the two systems recorded events at different times (for example, approval recorded in a batch after the seller already shipped). That is a hypothesis; the data cannot confirm which timestamp is right. We therefore keep both values and rely on the flag.

In [16]:
carrier_flagged = orders_prepared[orders_prepared["flag_carrier_before_approval"]]
carrier_gap_hours = (carrier_flagged["order_approved_at"]
                     - carrier_flagged["order_delivered_carrier_date"]).dt.total_seconds() / 3600
display(carrier_gap_hours.describe().round(1).to_frame("hours_carrier_before_approval"))

gap_bins = pd.cut(carrier_gap_hours, bins=[0, 1, 6, 24, 48, 24 * 7, float("inf")],
                  labels=["< 1 h", "1-6 h", "6-24 h", "1-2 days", "2-7 days", "> 7 days"],
                  include_lowest=True)
display(gap_bins.value_counts(sort=False).rename("orders").to_frame())
print("Order status of these orders:")
display(carrier_flagged["order_status"].value_counts().to_frame("orders"))

,hours_carrier_before_approval
count,1359.0
mean,24.8
std,115.3
min,0.0
25%,1.4
50%,17.2
75%,26.0
max,4109.3


,orders
< 1 h,242
1-6 h,379
6-24 h,280
1-2 days,280
2-7 days,164
> 7 days,14


Order status of these orders:


,orders
order_status,
delivered,1350
shipped,9


### 6.3 Actual delivery later than estimated delivery

Late delivery is a **delivery-performance outcome**, not invalid data. It is important for CallVerse because late orders are a plausible reason for future (simulated) "where is my order?" contacts.

`order_estimated_delivery_date` is a calendar date stored with the time `00:00:00`. Comparing it directly with the actual delivery *timestamp* would count an order delivered at 10:00 on the estimated day as late. We therefore compare **calendar dates**: an order is late when it was delivered on a day after the estimated day. The table below shows both comparisons so the difference from the Phase 1B count is visible.

In [17]:
estimate = orders_prepared["order_estimated_delivery_date"]
delivered = orders_prepared["order_delivered_customer_date"]

estimate_has_time = estimate.notna() & (estimate != estimate.dt.normalize())
print("Estimated dates that contain a time other than 00:00:00:", int(estimate_has_time.sum()))

comparable = delivered.notna() & estimate.notna()
late_by_timestamp = comparable & (delivered > estimate)
late_by_date = comparable & (delivered.dt.normalize() > estimate)

display(pd.DataFrame({
    "comparison": ["Delivery timestamp after estimate (Phase 1B method)",
                   "Delivery date after estimated date (used here)"],
    "comparable_orders": [int(comparable.sum()), int(comparable.sum())],
    "late_orders": [int(late_by_timestamp.sum()), int(late_by_date.sum())],
}))

# Nullable boolean: <NA> when there is no actual delivery date to compare
delivered_late = late_by_date.astype("boolean")
delivered_late[~comparable] = pd.NA
orders_prepared["delivered_after_estimated_date"] = delivered_late

days_late = (delivered.dt.normalize() - estimate).dt.days
print("Days late, for late orders only:")
display(days_late[late_by_date].describe().round(1).to_frame("days_late"))
display(orders_prepared.loc[late_by_date, ["order_id", "order_status", "order_estimated_delivery_date",
                                           "order_delivered_customer_date"]].head(5))

issue_log.append({"area": "Delivery performance", "dataset": "orders",
                  "issue": "Delivered on a day after the estimated date",
                  "affected_records": int(late_by_date.sum()),
                  "decision": "Outcome, not an error; stored as delivered_after_estimated_date (<NA> if not delivered)."})

Estimated dates that contain a time other than 00:00:00: 0


,comparison,comparable_orders,late_orders
0,Delivery timestamp after estimate (Phase 1B me...,96476,7827
1,Delivery date after estimated date (used here),96476,6535


Days late, for late orders only:


,days_late
count,6535.0
mean,10.6
std,14.6
min,1.0
25%,3.0
50%,7.0
75%,13.0
max,188.0


,order_id,order_status,order_estimated_delivery_date,order_delivered_customer_date
20,203096f03d82e0dffbc41ebc2e2bcfb7,delivered,2017-09-28,2017-10-09 22:23:46
25,fbf9ac61453ac646ce8ad9783d7d0af6,delivered,2018-03-12,2018-03-21 22:03:54
41,6ea2f835b4556291ffdc53fa0b3b95e8,delivered,2017-12-21,2017-12-28 18:59:23
57,66e4624ae69e7dc89bd50222b59f581f,delivered,2018-04-02,2018-04-03 13:28:46
58,a685d016c8a26f71a0bb67821070e398,delivered,2017-03-30,2017-04-06 13:37:16


### 6.4 Order status versus delivery timestamps

Some status/timestamp combinations are unusual: a `delivered` order without a customer delivery date, or a `canceled` order with one. These are flagged for review; the status and the timestamps are both left as they are.

In [18]:
status_timestamp_counts = orders_prepared.groupby("order_status")[ORDER_TIMESTAMP_COLUMNS].count()
status_timestamp_counts.insert(0, "orders", orders_prepared["order_status"].value_counts())
print("Non-missing timestamps per order status:")
display(status_timestamp_counts.sort_values("orders", ascending=False))

is_delivered = orders_prepared["order_status"] == "delivered"
is_canceled = orders_prepared["order_status"] == "canceled"
has_delivery_date = orders_prepared["order_delivered_customer_date"].notna()

orders_prepared["flag_delivered_status_without_delivery_date"] = is_delivered & ~has_delivery_date
orders_prepared["flag_canceled_status_with_delivery_date"] = is_canceled & has_delivery_date

for flag_name in ["flag_delivered_status_without_delivery_date", "flag_canceled_status_with_delivery_date"]:
    flagged_orders = orders_prepared[orders_prepared[flag_name]]
    print(f"{flag_name}: {len(flagged_orders)} orders")
    display(flagged_orders[["order_id", "order_status"] + ORDER_TIMESTAMP_COLUMNS].head(5))
    issue_log.append({"area": "Status consistency", "dataset": "orders", "issue": flag_name,
                      "affected_records": len(flagged_orders),
                      "decision": "Flagged; status and timestamps left unchanged."})

Non-missing timestamps per order status:


,orders,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,,
delivered,96478,96478,96464,96476,96470,96478
shipped,1107,1107,1107,1107,0,1107
canceled,625,625,484,75,6,625
unavailable,609,609,609,0,0,609
invoiced,314,314,314,0,0,314
processing,301,301,301,0,0,301
created,5,5,0,0,0,5
approved,2,2,2,0,0,2


flag_delivered_status_without_delivery_date: 8 orders


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24


flag_canceled_status_with_delivery_date: 6 orders


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25


## 7. Missing values

### 7.1 Missing-value counts

Starting point from Phase 1B: orders miss some approval, carrier and delivery timestamps; reviews miss many titles and messages; products miss some category and physical attributes; the other tables had no missing values. We recompute the counts here (from the source columns, after timestamp parsing) rather than copying them from the report.

In [19]:
missing_rows = []
for dataset_name, dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    for column in dataset.columns:
        missing_count = int(prepared[column].isna().sum())
        missing_rows.append({"dataset": dataset_name, "column": column, "missing_values": missing_count,
                             "missing_percent": round(missing_count / len(prepared) * 100, 2)})
missing_df = pd.DataFrame(missing_rows)

print("Columns with missing values:")
display(missing_df[missing_df["missing_values"] > 0].sort_values(["dataset", "missing_percent"], ascending=[True, False]))

datasets_without_missing = []
for dataset_name in raw_datasets:
    if missing_df.loc[missing_df["dataset"] == dataset_name, "missing_values"].sum() == 0:
        datasets_without_missing.append(dataset_name)
print("Datasets with no missing values:", datasets_without_missing)

Columns with missing values:


,dataset,column,missing_values,missing_percent
6,orders,order_delivered_customer_date,2965,2.98
5,orders,order_delivered_carrier_date,1783,1.79
4,orders,order_approved_at,160,0.16
33,products,product_category_name,610,1.85
34,products,product_name_lenght,610,1.85
35,products,product_description_lenght,610,1.85
36,products,product_photos_qty,610,1.85
37,products,product_weight_g,2,0.01
38,products,product_length_cm,2,0.01
39,products,product_height_cm,2,0.01


Datasets with no missing values: ['order_items', 'customers', 'payments', 'sellers']


### 7.2 Missing order timestamps in context

Missing milestones are mostly expected: an order that is still `shipped` has no customer delivery date yet, and a `canceled` order may never reach the carrier. The table shows missing counts per status.

In [20]:
milestone_columns = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
missing_by_status = orders_prepared[milestone_columns].isna().groupby(orders_prepared["order_status"]).sum()
missing_by_status.insert(0, "orders", orders_prepared["order_status"].value_counts())
display(missing_by_status.sort_values("orders", ascending=False))

,orders,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


### 7.3 Missing-value decisions and availability flags

| Table | Missing fields | Decision |
|---|---|---|
| orders | approval, carrier handoff, customer delivery timestamps | Not imputed. Added `has_approval_timestamp`, `has_carrier_handoff_timestamp`, `has_customer_delivery_timestamp`. An order without a delivery date is never treated as delivered. |
| reviews | comment title, comment message | Not filled; empty text is a normal choice by the customer. Added `has_comment_title`, `has_comment_message`. |
| products | category, name/description length, photo count, weight, dimensions | Not filled; no product dropped. Added `has_category` and `has_complete_dimensions` (weight, length, height and width all present). |
| other tables | none | No action. |

All these flags are derived from the source columns and can be recomputed at any time, so the transformation is reversible.

In [21]:
orders_prepared["has_approval_timestamp"] = orders_prepared["order_approved_at"].notna()
orders_prepared["has_carrier_handoff_timestamp"] = orders_prepared["order_delivered_carrier_date"].notna()
orders_prepared["has_customer_delivery_timestamp"] = orders_prepared["order_delivered_customer_date"].notna()

reviews_prepared["has_comment_title"] = reviews_prepared["review_comment_title"].notna()
reviews_prepared["has_comment_message"] = reviews_prepared["review_comment_message"].notna()

dimension_columns = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
products_prepared["has_category"] = products_prepared["product_category_name"].notna()
products_prepared["has_complete_dimensions"] = products_prepared[dimension_columns].notna().all(axis=1)

# Check: are the products without a category the same ones missing name, description and photos?
descriptive_columns = ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
same_products = products_prepared[descriptive_columns].isna().all(axis=1).sum() == (~products_prepared["has_category"]).sum()
print("Products missing a category also miss name, description and photo count:", bool(same_products))

availability_summary = pd.DataFrame([
    {"dataset": "orders", "column": "has_approval_timestamp", "false_count": int((~orders_prepared["has_approval_timestamp"]).sum())},
    {"dataset": "orders", "column": "has_carrier_handoff_timestamp", "false_count": int((~orders_prepared["has_carrier_handoff_timestamp"]).sum())},
    {"dataset": "orders", "column": "has_customer_delivery_timestamp", "false_count": int((~orders_prepared["has_customer_delivery_timestamp"]).sum())},
    {"dataset": "reviews", "column": "has_comment_title", "false_count": int((~reviews_prepared["has_comment_title"]).sum())},
    {"dataset": "reviews", "column": "has_comment_message", "false_count": int((~reviews_prepared["has_comment_message"]).sum())},
    {"dataset": "products", "column": "has_category", "false_count": int((~products_prepared["has_category"]).sum())},
    {"dataset": "products", "column": "has_complete_dimensions", "false_count": int((~products_prepared["has_complete_dimensions"]).sum())},
])
display(availability_summary)

for _, row in availability_summary.iterrows():
    issue_log.append({"area": "Missing values", "dataset": row["dataset"], "issue": f"{row['column']} is False",
                      "affected_records": row["false_count"], "decision": "Not imputed; availability flag added."})

Products missing a category also miss name, description and photo count: True


,dataset,column,false_count
0,orders,has_approval_timestamp,160
1,orders,has_carrier_handoff_timestamp,1783
2,orders,has_customer_delivery_timestamp,2965
3,reviews,has_comment_title,87656
4,reviews,has_comment_message,58247
5,products,has_category,610
6,products,has_complete_dimensions,2


## 8. Orders without related records

### 8.1 Orders without items, payments, or reviews

We compare order IDs between tables using the original relationships. **A missing review is normal**: customers are not obliged to write one. Missing items or payments need context: the status breakdown below shows whether they are concentrated in orders that never completed (`unavailable`, `canceled`, `created`). Nothing is deleted or invented; we add `has_items`, `has_payments` and `has_reviews` to the prepared orders.

In [22]:
orders_prepared["has_items"] = orders_prepared["order_id"].isin(order_items_raw["order_id"])
orders_prepared["has_payments"] = orders_prepared["order_id"].isin(payments_raw["order_id"])
orders_prepared["has_reviews"] = orders_prepared["order_id"].isin(reviews_raw["order_id"])

total_orders = len(orders_prepared)
coverage_rows = []
for column, related_table in [("has_items", "order_items"), ("has_payments", "payments"), ("has_reviews", "reviews")]:
    without_count = int((~orders_prepared[column]).sum())
    coverage_rows.append({"related_table": related_table, "orders_without_records": without_count,
                          "percent_of_orders": round(without_count / total_orders * 100, 2)})
    issue_log.append({"area": "Coverage", "dataset": "orders", "issue": f"Orders without {related_table} records",
                      "affected_records": without_count, "decision": f"Kept; flagged with {column} = False."})
display(pd.DataFrame(coverage_rows))

,related_table,orders_without_records,percent_of_orders
0,order_items,775,0.78
1,payments,1,0.00
2,reviews,768,0.77


In [23]:
print("Order status of orders WITHOUT items:")
display(orders_prepared.loc[~orders_prepared["has_items"], "order_status"].value_counts().to_frame("orders"))

print("Order status of orders WITHOUT reviews:")
display(orders_prepared.loc[~orders_prepared["has_reviews"], "order_status"].value_counts().to_frame("orders"))

print("Orders WITHOUT payment records:")
display(orders_prepared.loc[~orders_prepared["has_payments"], ["order_id", "order_status"] + ORDER_TIMESTAMP_COLUMNS])

Order status of orders WITHOUT items:


,orders
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


Order status of orders WITHOUT reviews:


,orders
order_status,
delivered,646
shipped,75
canceled,20
unavailable,14
processing,6
invoiced,5
created,2


Orders WITHOUT payment records:


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


### 8.2 Payment totals versus item totals

As a cross-table consistency check, we compare each order's total payments with its item prices plus freight. Totals are computed only for this check; the prepared tables keep one row per item and per payment. Differences can be legitimate (for example, credit-card instalment interest or vouchers), so they are reported, not corrected.

In [24]:
item_value = order_items_raw["price"] + order_items_raw["freight_value"]
items_total_by_order = item_value.groupby(order_items_raw["order_id"]).sum()
payments_total_by_order = payments_raw.groupby("order_id")["payment_value"].sum()

totals = pd.DataFrame({"items_plus_freight": items_total_by_order, "payments": payments_total_by_order}).dropna()
totals["difference"] = totals["payments"] - totals["items_plus_freight"]
differs = totals["difference"].abs() > 1.0  # tolerance of 1 BRL for rounding

print("Orders with both items and payments:", len(totals))
print("Orders where payments differ from items + freight by more than 1 BRL:", int(differs.sum()))
display(totals.loc[differs, "difference"].describe().round(2).to_frame("payments_minus_items"))
display(totals[differs].head(5))

issue_log.append({"area": "Cross-table", "dataset": "payments vs order_items",
                  "issue": "Payment total differs from items + freight by > 1 BRL",
                  "affected_records": int(differs.sum()),
                  "decision": "Reported only; may reflect interest or vouchers. Unresolved."})

Orders with both items and payments: 98665
Orders where payments differ from items + freight by more than 1 BRL: 249


,payments_minus_items
count,249.00
mean,11.51
std,19.35
min,-51.62
25%,3.29
50%,6.98
75%,14.12
max,182.81


,items_plus_freight,payments,difference
order_id,,,
00789ce015e7e5791c7914f32bb4fad4,168.83,190.81,21.98
016726239765c18f66826453f39c64e3,235.13,265.77,30.64
01e51b7c3025655646143d09b911e1d7,33.10,35.02,1.92
028aa7c930356788f861ed1b7f984819,57.53,62.94,5.41
02f4dd90ba0feb8ec394cac05862d2b5,130.96,141.65,10.69


## 9. Complete the prepared datasets

The prepared copies already contain parsed timestamps, chronology and status flags, and availability flags. This section adds the remaining record-level flags for reviews and payments, then summarises every added column.

### 9.1 Review flags

- `flag_repeated_review_id`: the `review_id` appears in more than one row (section 5.4).
- `flag_order_has_multiple_reviews`: the order has more than one review row (section 5.5).
- `flag_review_created_before_purchase_date`: the review creation date is earlier than the purchase date of its order. Review creation dates have no time of day, so we compare calendar dates.

If a review written for an earlier order was also attached to a later order of the same person (section 5.4), it would look "created before purchase" for the later order. The cell below counts how many flagged reviews have a repeated `review_id`.

In [25]:
reviews_prepared["flag_repeated_review_id"] = reviews_prepared.duplicated(subset=["review_id"], keep=False)
reviews_prepared["flag_order_has_multiple_reviews"] = reviews_prepared.duplicated(subset=["order_id"], keep=False)

# Look up each review's purchase date by order_id (a lookup, not a join, so rows are not multiplied)
purchase_by_order = orders_prepared.set_index("order_id")["order_purchase_timestamp"]
review_purchase_date = reviews_prepared["order_id"].map(purchase_by_order).dt.normalize()
reviews_prepared["flag_review_created_before_purchase_date"] = (
    reviews_prepared["review_creation_date"] < review_purchase_date
)

flagged_reviews = reviews_prepared[reviews_prepared["flag_review_created_before_purchase_date"]]
print("Reviews created before the purchase date:", len(flagged_reviews))
example = flagged_reviews[["review_id", "order_id", "review_creation_date"]].copy()
example["order_purchase_timestamp"] = example["order_id"].map(purchase_by_order)
display(example.head(5))

# A review copied onto a later order of the same person would look "too early" for that order
print("...of which have a repeated review_id:", int(flagged_reviews["flag_repeated_review_id"].sum()))
print("...of which belong to an order with several reviews:", int(flagged_reviews["flag_order_has_multiple_reviews"].sum()))

issue_log.append({"area": "Timestamps", "dataset": "reviews", "issue": "Review created before purchase date",
                  "affected_records": len(flagged_reviews),
                  "decision": "Flagged; dates kept. Partly explained by repeated review_ids; rest unresolved."})

Reviews created before the purchase date: 64


,review_id,order_id,review_creation_date,order_purchase_timestamp
5293,3415c9f764e478409e8e0660ae816dd2,4d59cc2a2e7bb6c0a851725f5888a9b5,2018-07-31,2018-09-06 14:50:07
5503,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,2016-10-09,2016-10-22 08:25:27
8404,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,2018-08-03,2018-08-17 20:06:36
8486,f1f8696d256889c0fe9962a5d32746e0,392ed9afd714e3c74767d0c4d3e3f477,2018-08-26,2018-09-29 09:13:03
10412,da8e936bc2f1b52ec626f8aa41f3df44,0fb9645c128ac84788798f0129496ca7,2018-08-04,2018-08-21 10:15:51


...of which have a repeated review_id: 40
...of which belong to an order with several reviews: 3


### 9.2 Payment flags

Payment values of zero or less, zero instalments, and the payment type `not_defined` are unusual. They are flagged, not changed.

In [26]:
payments_prepared["flag_payment_type_not_defined"] = payments_prepared["payment_type"] == "not_defined"
payments_prepared["flag_non_positive_payment_value"] = payments_prepared["payment_value"] <= 0
payments_prepared["flag_zero_installments"] = payments_prepared["payment_installments"] == 0

display(payments_prepared["payment_type"].value_counts().to_frame("payments"))
for flag_name in ["flag_payment_type_not_defined", "flag_non_positive_payment_value", "flag_zero_installments"]:
    flagged_payments = payments_prepared[payments_prepared[flag_name]]
    print(f"{flag_name}: {len(flagged_payments)} payment rows")
    display(flagged_payments.head(5))
    issue_log.append({"area": "Payments", "dataset": "payments", "issue": flag_name,
                      "affected_records": len(flagged_payments), "decision": "Flagged; values unchanged."})

,payments
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


flag_payment_type_not_defined: 3 payment rows


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_type_not_defined,flag_non_positive_payment_value,flag_zero_installments
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,True,True,False
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,True,True,False
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,True,True,False


flag_non_positive_payment_value: 9 payment rows


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_type_not_defined,flag_non_positive_payment_value,flag_zero_installments
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,False,True,False
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,False,True,False
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,False,True,False
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,True,True,False
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,True,True,False


flag_zero_installments: 2 payment rows


,order_id,payment_sequential,payment_type,payment_installments,payment_value,flag_payment_type_not_defined,flag_non_positive_payment_value,flag_zero_installments
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,False,False,True
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,False,False,True


### 9.3 Summary of the prepared datasets

The next table lists, for each prepared dataset, the source columns whose type changed and the columns that were added. No source column was renamed or removed. (Column names such as `product_name_lenght` keep Olist's original spelling, so they still match the raw files.)

In [27]:
prepared_summary = []
for dataset_name, raw_dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    changed_types = []
    for column in raw_dataset.columns:
        if raw_dataset[column].dtype != prepared[column].dtype:
            changed_types.append(f"{column}: {raw_dataset[column].dtype} -> {prepared[column].dtype}")
    added_columns = [column for column in prepared.columns if column not in raw_dataset.columns]
    prepared_summary.append({"dataset": dataset_name, "rows": len(prepared),
                             "source_columns": len(raw_dataset.columns), "added_columns": len(added_columns),
                             "type_changes": "; ".join(changed_types) or "none",
                             "added": ", ".join(added_columns) or "none"})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(prepared_summary))

,dataset,rows,source_columns,added_columns,type_changes,added
0,orders,99441,8,13,order_purchase_timestamp: object -> datetime64[ns]; order_approved_at: object -> datetime64[ns]; order_delivered_carrier_date: object -> datetime64[ns]; order_delivered_customer_date: object -> datetime64[ns]; order_estimated_delivery_date: object -> datetime64[ns],"flag_approved_before_purchase, flag_carrier_before_approval, flag_delivered_before_carrier, flag_delivered_before_purchase, delivered_after_estimated_date, flag_delivered_status_without_delivery_date, flag_canceled_status_with_delivery_date, has_approval_timestamp, has_carrier_handoff_timestamp, has_customer_delivery_timestamp, has_items, has_payments, has_reviews"
1,order_items,112650,7,0,shipping_limit_date: object -> datetime64[ns],none
2,customers,99441,5,0,none,none
3,payments,103886,5,3,none,"flag_payment_type_not_defined, flag_non_positive_payment_value, flag_zero_installments"
4,reviews,99224,7,5,review_creation_date: object -> datetime64[ns]; review_answer_timestamp: object -> datetime64[ns],"has_comment_title, has_comment_message, flag_repeated_review_id, flag_order_has_multiple_reviews, flag_review_created_before_purchase_date"
5,products,32951,9,2,none,"has_category, has_complete_dimensions"
6,sellers,3095,4,0,none,none


In [28]:
FLAG_MEANINGS = {
    "flag_approved_before_purchase": "Approval timestamp earlier than purchase timestamp",
    "flag_carrier_before_approval": "Carrier handoff earlier than approval",
    "flag_delivered_before_carrier": "Customer delivery earlier than carrier handoff",
    "flag_delivered_before_purchase": "Customer delivery earlier than purchase (negative duration)",
    "delivered_after_estimated_date": "Delivered on a day after the estimated date (<NA> if not delivered)",
    "flag_delivered_status_without_delivery_date": "Status 'delivered' but no customer delivery timestamp",
    "flag_canceled_status_with_delivery_date": "Status 'canceled' but has a customer delivery timestamp",
    "has_approval_timestamp": "Approval timestamp present",
    "has_carrier_handoff_timestamp": "Carrier handoff timestamp present",
    "has_customer_delivery_timestamp": "Customer delivery timestamp present",
    "has_items": "Order has at least one item row",
    "has_payments": "Order has at least one payment row",
    "has_reviews": "Order has at least one review row",
    "has_comment_title": "Review has a title",
    "has_comment_message": "Review has a message",
    "flag_repeated_review_id": "review_id appears in more than one row",
    "flag_order_has_multiple_reviews": "Order has more than one review row",
    "flag_review_created_before_purchase_date": "Review creation date earlier than purchase date",
    "has_category": "Product has a category",
    "has_complete_dimensions": "Product has weight, length, height and width",
    "flag_payment_type_not_defined": "Payment type is 'not_defined'",
    "flag_non_positive_payment_value": "Payment value is zero or negative",
    "flag_zero_installments": "Payment has zero instalments",
}

flag_dictionary = []
for dataset_name, raw_dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    for column in prepared.columns:
        if column in raw_dataset.columns:
            continue
        flag_dictionary.append({"dataset": dataset_name, "column": column,
                                "meaning": FLAG_MEANINGS.get(column, "UNDOCUMENTED - add a meaning"),
                                "true_count": int(prepared[column].sum()),
                                "unknown_count": int(prepared[column].isna().sum())})
flag_dictionary_df = pd.DataFrame(flag_dictionary)
display(flag_dictionary_df)

,dataset,column,meaning,true_count,unknown_count
0,orders,flag_approved_before_purchase,Approval timestamp earlier than purchase timestamp,0,0
1,orders,flag_carrier_before_approval,Carrier handoff earlier than approval,1359,0
2,orders,flag_delivered_before_carrier,Customer delivery earlier than carrier handoff,23,0
3,orders,flag_delivered_before_purchase,Customer delivery earlier than purchase (negative duration),0,0
4,orders,delivered_after_estimated_date,Delivered on a day after the estimated date (<NA> if not delivered),6535,2965
5,orders,flag_delivered_status_without_delivery_date,Status 'delivered' but no customer delivery timestamp,8,0
6,orders,flag_canceled_status_with_delivery_date,Status 'canceled' but has a customer delivery timestamp,6,0
7,orders,has_approval_timestamp,Approval timestamp present,99281,0
8,orders,has_carrier_handoff_timestamp,Carrier handoff timestamp present,97658,0
9,orders,has_customer_delivery_timestamp,Customer delivery timestamp present,96476,0


## 10. Validate the prepared datasets

We repeat the relevant Phase 1B checks on the prepared copies. Remember: **a data-quality flag does not mean a record is unusable.** It marks records that later phases should consider before using a particular field.

### 10.1 Row and column counts

In [29]:
count_rows = []
for dataset_name, raw_dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    count_rows.append({"dataset": dataset_name, "raw_rows": len(raw_dataset), "prepared_rows": len(prepared),
                       "records_preserved": len(raw_dataset) == len(prepared),
                       "raw_columns": len(raw_dataset.columns), "prepared_columns": len(prepared.columns)})
count_df = pd.DataFrame(count_rows)
display(count_df)

if not count_df["records_preserved"].all():
    raise ValueError("A prepared dataset has a different number of rows than its source. Investigate before exporting.")
print("All record counts preserved: no row was added or removed.")

,dataset,raw_rows,prepared_rows,records_preserved,raw_columns,prepared_columns
0,orders,99441,99441,True,8,21
1,order_items,112650,112650,True,7,7
2,customers,99441,99441,True,5,5
3,payments,103886,103886,True,5,8
4,reviews,99224,99224,True,7,12
5,products,32951,32951,True,9,11
6,sellers,3095,3095,True,4,4


All record counts preserved: no row was added or removed.


### 10.2 Duplicates and identifier uniqueness

Exact duplicates are checked on the source columns only (the added flags could hide or create differences).

In [30]:
validation_keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "customers": ["customer_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

uniqueness_rows = []
for dataset_name, raw_dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    source_columns = list(raw_dataset.columns)
    key_columns = validation_keys[dataset_name]
    uniqueness_rows.append({"dataset": dataset_name,
                            "exact_duplicate_rows": int(prepared[source_columns].duplicated().sum()),
                            "record_key": ", ".join(key_columns),
                            "key_is_unique": not prepared.duplicated(subset=key_columns).any(),
                            "repeated_review_id_rows": int(prepared["flag_repeated_review_id"].sum()) if dataset_name == "reviews" else None})
display(pd.DataFrame(uniqueness_rows))

,dataset,exact_duplicate_rows,record_key,key_is_unique,repeated_review_id_rows
0,orders,0,order_id,True,NaN
1,order_items,0,"order_id, order_item_id",True,NaN
2,customers,0,customer_id,True,NaN
3,payments,0,"order_id, payment_sequential",True,NaN
4,reviews,0,"review_id, order_id",True,1603.0
5,products,0,product_id,True,NaN
6,sellers,0,seller_id,True,NaN


### 10.3 Missing values and timestamp parsing

The source columns of each prepared dataset should have exactly the same number of missing values as the raw data. If a timestamp had failed to parse, it would appear here as an extra missing value.

In [31]:
missing_comparison = []
for dataset_name, raw_dataset in raw_datasets.items():
    prepared = prepared_datasets[dataset_name]
    for column in raw_dataset.columns:
        raw_missing = int(raw_dataset[column].isna().sum())
        prepared_missing = int(prepared[column].isna().sum())
        if raw_missing > 0 or prepared_missing > 0:
            missing_comparison.append({"dataset": dataset_name, "column": column, "raw_missing": raw_missing,
                                       "prepared_missing": prepared_missing, "unchanged": raw_missing == prepared_missing})
missing_comparison_df = pd.DataFrame(missing_comparison)
display(missing_comparison_df)
print("Missing values unchanged in every source column:", bool(missing_comparison_df["unchanged"].all()))

print("Timestamp parse failures:")
display(parsing_results_df[["dataset", "column", "parse_failures"]])

,dataset,column,raw_missing,prepared_missing,unchanged
0,orders,order_approved_at,160,160,True
1,orders,order_delivered_carrier_date,1783,1783,True
2,orders,order_delivered_customer_date,2965,2965,True
3,reviews,review_comment_title,87656,87656,True
4,reviews,review_comment_message,58247,58247,True
5,products,product_category_name,610,610,True
6,products,product_name_lenght,610,610,True
7,products,product_description_lenght,610,610,True
8,products,product_photos_qty,610,610,True
9,products,product_weight_g,2,2,True


Missing values unchanged in every source column: True
Timestamp parse failures:


,dataset,column,parse_failures
0,orders,order_purchase_timestamp,0
1,orders,order_approved_at,0
2,orders,order_delivered_carrier_date,0
3,orders,order_delivered_customer_date,0
4,orders,order_estimated_delivery_date,0
5,order_items,shipping_limit_date,0
6,reviews,review_creation_date,0
7,reviews,review_answer_timestamp,0


### 10.4 Referential integrity

In [32]:
relationships = [
    ("Orders to customers", orders_prepared, "customer_id", customers_prepared, "customer_id"),
    ("Items to orders", order_items_prepared, "order_id", orders_prepared, "order_id"),
    ("Items to products", order_items_prepared, "product_id", products_prepared, "product_id"),
    ("Items to sellers", order_items_prepared, "seller_id", sellers_prepared, "seller_id"),
    ("Payments to orders", payments_prepared, "order_id", orders_prepared, "order_id"),
    ("Reviews to orders", reviews_prepared, "order_id", orders_prepared, "order_id"),
]

integrity_rows = []
for label, child, child_key, parent, parent_key in relationships:
    unmatched = child[child_key].notna() & ~child[child_key].isin(parent[parent_key])
    integrity_rows.append({"relationship": label, "child_rows": len(child),
                           "missing_child_keys": int(child[child_key].isna().sum()),
                           "unmatched_references": int(unmatched.sum())})
    issue_log.append({"area": "Referential integrity", "dataset": label, "issue": "Unmatched references",
                      "affected_records": int(unmatched.sum()), "decision": "Checked on prepared data."})
display(pd.DataFrame(integrity_rows))

,relationship,child_rows,missing_child_keys,unmatched_references
0,Orders to customers,99441,0,0
1,Items to orders,112650,0,0
2,Items to products,112650,0,0
3,Items to sellers,112650,0,0
4,Payments to orders,103886,0,0
5,Reviews to orders,99224,0,0


### 10.5 Timestamp chronology

We recount the chronology violations directly from the prepared timestamps and compare them with the stored flags.

In [33]:
chronology_validation = []
for flag_name, earlier_column, later_column in chronology_checks:
    recount = int((orders_prepared[later_column] < orders_prepared[earlier_column]).sum())  # NaT comparisons are False
    stored = int(orders_prepared[flag_name].sum())
    chronology_validation.append({"flag": flag_name, "recounted_violations": recount,
                                  "flagged_records": stored, "match": recount == stored})
display(pd.DataFrame(chronology_validation))

,flag,recounted_violations,flagged_records,match
0,flag_approved_before_purchase,0,0,True
1,flag_carrier_before_approval,1359,1359,True
2,flag_delivered_before_carrier,23,23,True
3,flag_delivered_before_purchase,0,0,True


### 10.6 The raw DataFrames were not modified

In [34]:
fingerprint_rows = []
for dataset_name, dataset in raw_datasets.items():
    current = pd.util.hash_pandas_object(dataset, index=True).sum()
    fingerprint_rows.append({"dataset": dataset_name, "raw_unchanged": current == raw_fingerprints[dataset_name]})
fingerprint_df = pd.DataFrame(fingerprint_rows)
display(fingerprint_df)
if not fingerprint_df["raw_unchanged"].all():
    raise ValueError("A raw DataFrame was modified during preparation.")

,dataset,raw_unchanged
0,orders,True
1,order_items,True
2,customers,True
3,payments,True
4,reviews,True
5,products,True
6,sellers,True


## 11. Export the prepared datasets

**Format.** The project's data dictionary (`docs/data_dictionary.md`) plans processed data as Parquet files in `data/processed/`, and `pyarrow` is a core project dependency. We follow that convention. Unlike CSV, Parquet stores column types, so the datetime columns, text ZIP codes and nullable booleans are loaded back with the right types. (If a CSV copy is ever needed, remember that CSV stores timestamps as text: they must be parsed again with `pd.to_datetime` after loading, and ZIP codes must be read with `dtype=str`.)

**Structure.** One file per table, named `olist_<table>_prepared.parquet` to distinguish them from the raw `olist_*_dataset.csv` files. No flattened, joined table is exported. The issue log is exported as a small CSV for reference.

`data/processed/` already exists in the project (tracked with a `.gitkeep`); its contents are ignored by git, so exported files stay local.

In [35]:
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"Expected processed-data directory {PROCESSED_DIR}. It is part of the project structure (data/processed/.gitkeep).")

exported_files = []
for dataset_name, prepared in prepared_datasets.items():
    output_path = PROCESSED_DIR / f"olist_{dataset_name}_prepared.parquet"
    prepared.to_parquet(output_path, index=False)
    exported_files.append(output_path)

issue_log_df = pd.DataFrame(issue_log)
issue_log_path = PROCESSED_DIR / "olist_preparation_issue_log.csv"
issue_log_df.to_csv(issue_log_path, index=False)
exported_files.append(issue_log_path)

# Read each Parquet file back and confirm rows, columns and types survived
export_check = []
for dataset_name, prepared in prepared_datasets.items():
    output_path = PROCESSED_DIR / f"olist_{dataset_name}_prepared.parquet"
    reloaded = pd.read_parquet(output_path)
    export_check.append({"file": output_path.name, "rows": len(reloaded), "columns": len(reloaded.columns),
                         "same_rows": len(reloaded) == len(prepared),
                         "same_types": reloaded.dtypes.equals(prepared.dtypes),
                         "size_mb": round(output_path.stat().st_size / 1024**2, 2)})
display(pd.DataFrame(export_check))
print("Issue log:", issue_log_path.relative_to(PROJECT_ROOT))

,file,rows,columns,same_rows,same_types,size_mb
0,olist_orders_prepared.parquet,99441,21,True,True,10.06
1,olist_order_items_prepared.parquet,112650,7,True,True,6.21
2,olist_customers_prepared.parquet,99441,5,True,True,6.78
3,olist_payments_prepared.parquet,103886,8,True,True,3.70
4,olist_reviews_prepared.parquet,99224,12,True,True,9.21
5,olist_products_prepared.parquet,32951,11,True,True,1.36
6,olist_sellers_prepared.parquet,3095,4,True,True,0.13


Issue log: data\processed\olist_preparation_issue_log.csv


## 12. Summary of cleaning decisions and limitations

The issue log below collects every count computed in this notebook, with the decision taken.

In [36]:
display(issue_log_df)

,area,dataset,issue,affected_records,decision
0,Duplicates,orders,Exact duplicate rows,0,Reported; no rows removed.
1,Duplicates,order_items,Exact duplicate rows,0,Reported; no rows removed.
2,Duplicates,customers,Exact duplicate rows,0,Reported; no rows removed.
3,Duplicates,payments,Exact duplicate rows,0,Reported; no rows removed.
4,Duplicates,reviews,Exact duplicate rows,0,Reported; no rows removed.
5,Duplicates,products,Exact duplicate rows,0,Reported; no rows removed.
6,Duplicates,sellers,Exact duplicate rows,0,Reported; no rows removed.
7,Identifiers,customers,customer_unique_id repeats across orders (repeat customers),2997,Expected behaviour; use customer_unique_id to identify people.
8,Identifiers,reviews,"review_id repeated across rows (identical content, same person)",1603,"Kept all rows; key is (review_id, order_id); flagged flag_repeated_review_id."
9,Identifiers,reviews,Orders with more than one review row,547,Kept all rows; flagged flag_order_has_multiple_reviews. Which review is final is unresolved.


**Transformations performed**

- Read ZIP-code prefixes as text, exactly as written in the CSV files (keeps leading zeros).
- Converted order milestones, the item shipping limit, and the two review dates to datetime using the explicit Olist format.
- Added documented `has_*` availability columns and `flag_*` data-quality columns (see the flag dictionary in 9.3), plus `delivered_after_estimated_date` as a nullable delivery-performance outcome.
- No source column was renamed, removed, filled, or corrected.

**Records preserved or removed**

- Every record of every table was preserved (section 10.1). No duplicates were removed because there are no exact duplicate rows, and repeated `review_id` rows are legitimate links between one review and several orders.

**Missing values**

- Missing timestamps, review texts and product attributes were left missing. Availability flags make them easy to filter later. Undelivered orders are never treated as delivered.

**Identifier issues that remain unresolved**

- Repeated `review_id` values: interpreted as one review attached to several orders of the same person, but not confirmed by documentation. Use `(review_id, order_id)` as the key and `flag_repeated_review_id` to avoid double counting.
- Orders with several review rows: which review is the final one is undecided.
- People must be identified with `customer_unique_id`, not `customer_id`.

**Timestamp inconsistencies that remain unresolved**

- Carrier handoff before approval, delivery before carrier handoff, and reviews created before the purchase date are flagged but not corrected; the data do not show which value is wrong. Part of the early reviews coincide with repeated `review_id` values (section 9.1); the others remain unexplained.
- `delivered`/`canceled` orders with unexpected delivery timestamps are flagged.
- Late delivery is measured by calendar date, which gives a lower count than the timestamp comparison used in Phase 1B.

**Referential integrity**

- All child keys match their parent tables (section 10.4). Some orders have no items, payments or reviews (section 8); they are kept and marked with `has_items`, `has_payments`, `has_reviews`. Some payment totals differ from item totals; this is reported, not resolved.

**Files exported** (in `data/processed/`)

- `olist_orders_prepared.parquet`, `olist_order_items_prepared.parquet`, `olist_customers_prepared.parquet`, `olist_payments_prepared.parquet`, `olist_reviews_prepared.parquet`, `olist_products_prepared.parquet`, `olist_sellers_prepared.parquet`
- `olist_preparation_issue_log.csv`

**Limitations of Olist for call-center simulation**

- Olist records orders, deliveries, payments and reviews. It contains **no customer contacts, contact reasons, channels, call durations, queue waiting times, abandonment, or agent assignments**.
- A late delivery, cancellation or low review score can motivate a *simulated* contact scenario, but does not prove that the customer contacted support.
- Reviews are post-purchase feedback, not contact-center surveys or transcripts.
- Later features must use only information available at the simulated contact time (for example, an actual delivery date is not known before it happens).
- Product category names are in Portuguese; the translation file was not part of this phase, and some categories have no translation.

> **Synthetic contact records will be generated in a later phase.** They will be based on explicit, documented assumptions and must never be confused with observed Olist data. This notebook does not generate any contact data.